In [0]:
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown(
    "schema",
    "migration_dev",
    ["migration_dev", "migration_test", "migration_prod_sim"],
)
dbutils.widgets.text("batch_id", "batch_001")

catalog_name = dbutils.widgets.get("catalog").strip()
schema_name = dbutils.widgets.get("schema").strip()
batch_id = dbutils.widgets.get("batch_id").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog_name):
    raise ValueError("Invalid catalog name")

if schema_name not in {
    "migration_dev", "migration_test", "migration_prod_sim"
}:
    raise ValueError("Invalid environment schema")

if not re.fullmatch(r"batch_[0-9]+", batch_id):
    raise ValueError("Invalid batch ID")

table_prefix = f"{catalog_name}.{schema_name}"
volume_root = f"/Volumes/{catalog_name}/{schema_name}"

print(f"Environment: {table_prefix}")
print(f"Batch: {batch_id}")
print(f"Volume root: {volume_root}")

In [0]:
from pyspark.sql import functions as F

 

control_rows = (
    spark.table(f"{table_prefix}.batch_control")
    .filter(F.col("batch_id") == batch_id)
    .collect()
)

if len(control_rows) != 1:
    raise RuntimeError("Expected exactly one batch control record")

control = control_rows[0]

if control["batch_status"] != "APPROVED" or control["failed_checks"] != 0:
    raise RuntimeError("Batch is not approved; Gold publication blocked")

silver = (
    spark.table(f"{table_prefix}.silver_documents")
    .filter(F.col("batch_id") == batch_id)
)

# Silver must belong to the current approval run.
if silver.filter(
    ~F.col("validation_run_id").eqNullSafe(control["validation_run_id"])
).limit(1).count() > 0:
    raise RuntimeError("Silver approval is outdated; rerun the Silver notebook")

if silver.count() != control["expected_document_count"]:
    raise RuntimeError("Silver count differs from the approved batch")

if silver.groupBy("document_id").count().filter(
    F.col("count") > 1
).limit(1).count() > 0:
    raise RuntimeError("Duplicate document IDs; Gold publication blocked")

gold = silver.select(
    "batch_id",
    "document_id",
    "account_number",
    "document_type",
    "statement_date",
    "file_name",
    "pdf_path",
    "source_sha256",
    "file_size_bytes",
    "validation_run_id",
    F.current_timestamp().alias("published_at"),
)

gold.createOrReplaceTempView("current_gold_batch")

spark.sql(f"""
    MERGE INTO {table_prefix}.gold_load_manifest AS target
    USING current_gold_batch AS source
    ON target.batch_id = source.batch_id
       AND target.document_id = source.document_id
    WHEN MATCHED
         AND target.validation_run_id <> source.validation_run_id
         THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

saved = (
    spark.table(f"{table_prefix}.gold_load_manifest")
    .filter(F.col("batch_id") == batch_id)
)

print(f"Gold loading instructions: {saved.count()}")
display(saved.orderBy("document_id").limit(5))